# Structured extraction with Tiny Aya: baseline, fine-tune, re-evaluate

Pipeline: CuratorKIT-curated data -> AlignTune LoRA SFT -> field-level evaluation.
Needs a GPU, internet, and a Kaggle secret named `HF_TOKEN` (a Hugging Face token whose account has accepted the Tiny Aya license).

In [ ]:
!pip install -q aligntune peft bitsandbytes accelerate jsonschema huggingface_hub 2>&1 | tail -2

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download
token = UserSecretsClient().get_secret('HF_TOKEN').strip().strip('"').strip("'")
os.environ['HF_TOKEN'] = token
login(token=token)  # also saved on disk for the training and evaluation subprocesses
print('logged in as', HfApi().whoami(token=token)['name'])
snapshot_download('iamvisheshsrivastava/lexsi-extraction-data', repo_type='dataset', local_dir='/kaggle/working/data/final', token=token)
print(os.listdir('/kaggle/working/data/final'))

In [ ]:
%%writefile /kaggle/working/schema.py
"""The extraction target: one flat JSON object per support ticket or invoice."""

DOC_TYPES = ["support_ticket", "invoice"]
LANGUAGES = {"en": "English", "de": "German", "fr": "French", "hi": "Hinglish (Hindi written in Latin script mixed with English)"}
CATEGORIES = ["billing", "shipping", "technical", "account", "refund", "other"]
PRIORITIES = ["low", "medium", "high", "urgent"]

FIELDS = ["doc_type", "language", "party_name", "reference_id", "product", "issue_category", "priority",
          "amount", "currency", "date_iso"]

SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": FIELDS,
    "properties": {
        "doc_type": {"enum": DOC_TYPES},
        "language": {"enum": list(LANGUAGES)},
        "party_name": {"type": ["string", "null"]},
        "reference_id": {"type": ["string", "null"]},
        "product": {"type": ["string", "null"]},
        "issue_category": {"enum": CATEGORIES},
        "priority": {"enum": PRIORITIES},
        "amount": {"type": ["number", "null"]},
        "currency": {"type": ["string", "null"], "pattern": "^[A-Z]{3}$"},
        "date_iso": {"type": ["string", "null"], "pattern": "^\\d{4}-\\d{2}-\\d{2}$"},
    },
}

INSTRUCTION = (
    "Extract the information from the document below into a single JSON object with exactly these keys: "
    + ", ".join(FIELDS) + ". Use null for anything not stated, except doc_type, language, issue_category and priority, "
    "which must always be one of the allowed values and never null. "
    "doc_type is support_ticket or invoice; language is en, de, fr or hi (hi = Hinglish); "
    "issue_category is one of " + "/".join(CATEGORIES) + "; priority is one of " + "/".join(PRIORITIES) + "; "
    "amount is a number; currency is a 3-letter ISO code; date_iso is YYYY-MM-DD. Output only the JSON."
)


In [ ]:
%%writefile /kaggle/working/eval_extraction.py
"""Metrics for structured-extraction outputs: JSON validity, schema pass, field F1 and exact match.

Predictions and references are dicts keyed by example id. A prediction is the raw model text; the
reference is the gold JSON object. Usage as a module: evaluate(predictions, references, schema).
"""
import json
import re

from jsonschema import Draft202012Validator


def parse_json(text):
    """Return the first JSON object in text, or None."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        start = text.find("{")
        while start != -1:
            depth = 0
            for end in range(start, len(text)):
                depth += text[end] == "{"
                depth -= text[end] == "}"
                if depth == 0:
                    try:
                        value = json.loads(text[start:end + 1])
                        return value if isinstance(value, dict) else None
                    except json.JSONDecodeError:
                        break
            start = text.find("{", start + 1)
    return None


def normalise(value):
    if isinstance(value, str):
        return re.sub(r"\s+", " ", value).strip().casefold()
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value


def field_scores(prediction, reference):
    """Return (true positives, predicted fields, reference fields) over top-level key/value pairs."""
    gold = {k: normalise(v) for k, v in reference.items() if v not in (None, "", [])}
    pred = {k: normalise(v) for k, v in (prediction or {}).items() if v not in (None, "", [])}
    tp = sum(1 for k, v in pred.items() if gold.get(k) == v)
    return tp, len(pred), len(gold)


def evaluate(predictions, references, schema):
    validator = Draft202012Validator(schema)
    n = len(references)
    valid_json = schema_pass = exact = 0
    tp = n_pred = n_gold = 0
    for key, reference in references.items():
        parsed = parse_json(predictions.get(key, ""))
        valid_json += parsed is not None
        if parsed is not None and not list(validator.iter_errors(parsed)):
            schema_pass += 1
        a, b, c = field_scores(parsed, reference)
        tp, n_pred, n_gold = tp + a, n_pred + b, n_gold + c
        exact += parsed is not None and a == b == c
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gold if n_gold else 0.0
    return {
        "n": n,
        "valid_json_rate": valid_json / n,
        "schema_pass_rate": schema_pass / n,
        "field_precision": precision,
        "field_recall": recall,
        "field_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "exact_match_rate": exact / n,
    }


In [ ]:
%%writefile /kaggle/working/eval_model.py
"""Evaluate a model (optionally with a LoRA adapter) on the extraction test set.

Reports valid-JSON rate, schema pass rate, field F1, exact match (overall and per language / doc type)
plus generation throughput (for the ROI comparison). Meant for a GPU session (Kaggle / Colab).

Run: python eval_model.py --model CohereLabs/tiny-aya-global --data data/final --out results/baseline.json
     python eval_model.py --model CohereLabs/tiny-aya-global --adapter out/adapter --out results/tuned.json
"""
import argparse
import json
import time
from collections import defaultdict
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from eval_extraction import evaluate
from schema import INSTRUCTION, SCHEMA


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--adapter")
    parser.add_argument("--data", default="data/final")
    parser.add_argument("--split", default="test")
    parser.add_argument("--out", required=True)
    parser.add_argument("--limit", type=int)
    parser.add_argument("--batch", type=int, default=8)
    parser.add_argument("--max_new_tokens", type=int, default=256)
    parser.add_argument("--no-stop", action="store_true", help="do not stop at the closing brace (the first evaluation run did not)")
    parser.add_argument("--prompt", choices=["short", "full"], default="short",
                        help="short = the minimal prompt used for training; full = the full schema instruction")
    args = parser.parse_args()

    rows = [json.loads(line) for line in open(Path(args.data) / f"{args.split}.jsonl", encoding="utf-8")]
    rows = rows[: args.limit] if args.limit else rows
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model, device_map="auto", torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type="nf4"))
    if args.adapter:
        model = PeftModel.from_pretrained(model, args.adapter)
    model.eval()

    # Stop when the model ends its turn, ends its response, or closes the (flat) JSON object. Without this a few
    # outputs ran to the token limit and held up their whole batch in the first evaluation run.
    end_response = tokenizer.convert_tokens_to_ids("<|END_RESPONSE|>")
    eos_ids = sorted({i for i in (tokenizer.eos_token_id, end_response) if isinstance(i, int) and i != tokenizer.unk_token_id})
    gen_kwargs = dict(max_new_tokens=args.max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids)
    if not args.no_stop:
        gen_kwargs.update(stop_strings=["}"], tokenizer=tokenizer)
    predictions, generated_tokens, at_limit, start = {}, 0, 0, time.time()
    for i in range(0, len(rows), args.batch):
        chunk = rows[i:i + args.batch]
        texts = [tokenizer.apply_chat_template([{"role": "user", "content": r["prompt"] if args.prompt == "short" else
                                                 INSTRUCTION + "\n\nDocument:\n" + r["prompt"].split("Document:\n", 1)[1]}], tokenize=False,
                                               add_generation_prompt=True) for r in chunk]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            try:
                output = model.generate(**inputs, **gen_kwargs)
            except (TypeError, ValueError) as error:  # stop_strings unsupported by this transformers version: fall back
                if "stop_strings" not in gen_kwargs:
                    raise
                print("warning: stop_strings unavailable, generating without it:", str(error)[:120], flush=True)
                gen_kwargs.pop("stop_strings")
                gen_kwargs.pop("tokenizer")
                output = model.generate(**inputs, **gen_kwargs)
        for j, row in enumerate(chunk):
            new = output[j][inputs["input_ids"].shape[1]:]
            at_limit += int((new != tokenizer.pad_token_id).sum() >= args.max_new_tokens)
            generated_tokens += int((new != tokenizer.pad_token_id).sum())
            predictions[i + j] = tokenizer.decode(new, skip_special_tokens=True)
        print(f"[eval] {min(i + args.batch, len(rows))}/{len(rows)} done, {time.time() - start:.0f}s elapsed", flush=True)
    seconds = time.time() - start

    references = {i: json.loads(r["output"]) for i, r in enumerate(rows)}
    result = {"model": args.model, "adapter": args.adapter, "split": args.split, "prompt": args.prompt,
              "stop_at_brace": not args.no_stop, "outputs_at_token_limit": at_limit, "n_documents": len(rows),
              "overall": evaluate(predictions, references, SCHEMA),
              "tokens_per_second": generated_tokens / seconds, "seconds": seconds}
    for field in ("language", "doc_type"):
        groups = defaultdict(list)
        for i, r in enumerate(rows):
            groups[r[field]].append(i)
        result["by_" + field] = {g: evaluate({i: predictions[i] for i in idx}, {i: references[i] for i in idx}, SCHEMA)
                                 for g, idx in groups.items()}
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    Path(args.out).with_suffix(".predictions.json").write_text(json.dumps(predictions, indent=1, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(result["overall"], indent=2))


if __name__ == "__main__":
    main()


## Baseline: untuned model with the full schema prompt

The pipeline notebook evaluated the untuned model with the minimal prompt used for fine-tuning, which never names the JSON keys. This run gives the untuned model the full schema instruction, for a fair comparison.

In [ ]:
%cd /kaggle/working
!python eval_model.py --model CohereLabs/tiny-aya-global --data data/final --prompt full --out results/baseline_full.json

In [ ]:
import json
print(json.dumps(json.load(open('results/baseline_full.json'))['overall'], indent=1))